# Sweep 15a — Loss Functions

**Question:** Which loss formulation maximises Jaccard on the full MIRROR system?

**Design:** 11 loss variants × 3 seeds = 33 runs ~5h (full system only)

| Config | Change from baseline |
|--------|---------------------|
| L0_baseline | 0.5×BCE + 1.0×SoftJac + 0.05×margin (current) |
| L1_bce_only | SoftJac=0, margin=0 |
| L2_jac_only | BCE=0, margin=0 |
| L3_bce_heavy | 1.0×BCE + 0.5×SoftJac |
| L4_jac_heavy | 0.3×BCE + 1.5×SoftJac |
| L5_no_margin | margin=0 |
| L6_hi_margin | margin=0.2 |
| L7_focal | Focal loss gamma_neg=2 |
| L8_smooth | label_smoothing=0.1 |
| L9_atc | + ATC coarse auxiliary (weight=0.1) |
| L10_lab_impute | + Lab imputation auxiliary |

**Locked-in:** aggregator=last, ar_seq_len=20, note_proj=128, encoder=transformer, fusion=film

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn


In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    for find_file in ["cohort_mimic3.pkl", "records_final.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            break

    emb_paths = glob.glob("/kaggle/input/**/code_embeddings.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    note_paths = glob.glob("/kaggle/input/**/note_embeddings_mimic3.pkl", recursive=True)
    if note_paths:
        note_src = note_paths[0]
        note_link = "./data/processed/note_embeddings_mimic3.pkl"
        if not os.path.exists(note_link):
            os.symlink(note_src, note_link)
        print(f"Note embeddings linked: {note_src}")
    else:
        print("WARNING: note_embeddings_mimic3.pkl not found — notes contexts will fail")

    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

    # Patch DrugGNN to accept ehr_adj (model.py passes it to all graph encoders;
    # only hgt_ehr_feat uses it — DrugGNN must absorb and ignore it).
    gnn_path = "/kaggle/working/src/model/graph_encoders/drug_gnn.py"
    if os.path.exists(gnn_path):
        with open(gnn_path, "r") as rf:
            content = rf.read()
        if "ehr_adj" not in content:
            old = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n    ):"
            new = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n        ehr_adj=None,  # absorbed — only hgt_ehr_feat uses this\n        **kwargs,\n    ):"
            if old in content:
                content = content.replace(old, new)
                with open(gnn_path, "w") as wf:
                    wf.write(content)
                print("  drug_gnn.py patched (ehr_adj absorbed)")
            else:
                print("  WARNING: drug_gnn.py patch target not found — check manually")

    # Patch DCMAScorer: add nn.Linear projections so notes_repr (768d) and
    # labs_repr (400d) are both projected to hidden_dim before torch.cat(dim=1).
    # Uses regular Linear (not LazyLinear) so parameter counting at init is safe.
    dcma_path = "/kaggle/working/src/model/decoders/dcma_decoder.py"
    if os.path.exists(dcma_path):
        with open(dcma_path, "r") as rf:
            content = rf.read()
        if "note_proj" not in content:
            content = content.replace(
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)",
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3,\n"
                "                 note_repr_dim: int = 768, lab_repr_dim: int = 400, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)\n"
                "        self.hidden_dim = hidden_dim\n"
                "        self.note_proj = nn.Linear(note_repr_dim, hidden_dim)\n"
                "        self.lab_proj  = nn.Linear(lab_repr_dim,  hidden_dim)",
            )
            content = content.replace(
                "            tokens.append(F.normalize(notes_repr, dim=-1).unsqueeze(1))",
                "            tokens.append(self.note_proj(F.normalize(notes_repr, dim=-1)).unsqueeze(1))",
            )
            content = content.replace(
                "            tokens.append(F.normalize(labs_repr, dim=-1).unsqueeze(1))",
                "            tokens.append(self.lab_proj(F.normalize(labs_repr, dim=-1)).unsqueeze(1))",
            )
            with open(dcma_path, "w") as wf:
                wf.write(content)
            print("  dcma_decoder.py patched (note_proj + lab_proj added)")
        else:
            print("  dcma_decoder.py already patched")

print("Setup done:", os.getcwd())


In [ ]:
import yaml, subprocess, gc, json, numpy as np
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, training_overrides=None,
                preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if training_overrides:
        for k, v in training_overrides.items():
            cfg["training"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)


# ── Locked-in best config (sweeps 11a-13a) ────────────────────────────────
BEST_MODEL = {
    "ar_max_seq_len": 20,
    "note_proj_dim": 128,
    "graph_encoder_type": "drug_gnn",
    "graph_layer_type": "hgt",
    "hgt_layers": 2,
}
BACKBONE_FLAGS = (
    "--device cuda "
    "--visit_level_training "
    "--fusion_strategy film "
    "--aggregator_type last "
)
LAB_FLAGS = "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200"
FULL_PREP  = {"lab_dim": 400}
# SEEDS defined here so LOSS_CONFIGS print below can reference it
SEEDS = [42, 123, 456]

# (name, training_overrides_dict, extra_cli_flags)
LOSS_CONFIGS = [
    ("L0_baseline",
     {"bce_weight": 0.5,  "soft_jaccard_weight": 1.0, "margin_weight": 0.05,
      "label_smoothing": 0.0, "use_focal": False}, ""),
    ("L1_bce_only",
     {"bce_weight": 1.0,  "soft_jaccard_weight": 0.0, "margin_weight": 0.0,
      "label_smoothing": 0.0, "use_focal": False}, ""),
    ("L2_jac_only",
     {"bce_weight": 0.0,  "soft_jaccard_weight": 1.0, "margin_weight": 0.0,
      "label_smoothing": 0.0, "use_focal": False}, ""),
    ("L3_bce_heavy",
     {"bce_weight": 1.0,  "soft_jaccard_weight": 0.5, "margin_weight": 0.05,
      "label_smoothing": 0.0, "use_focal": False}, ""),
    ("L4_jac_heavy",
     {"bce_weight": 0.3,  "soft_jaccard_weight": 1.5, "margin_weight": 0.05,
      "label_smoothing": 0.0, "use_focal": False}, ""),
    ("L5_no_margin",
     {"bce_weight": 0.5,  "soft_jaccard_weight": 1.0, "margin_weight": 0.0,
      "label_smoothing": 0.0, "use_focal": False}, ""),
    ("L6_hi_margin",
     {"bce_weight": 0.5,  "soft_jaccard_weight": 1.0, "margin_weight": 0.2,
      "label_smoothing": 0.0, "use_focal": False}, ""),
    ("L7_focal",
     {"bce_weight": 0.5,  "soft_jaccard_weight": 1.0, "margin_weight": 0.05,
      "label_smoothing": 0.0,
      "use_focal": True, "focal_gamma_neg": 2.0, "focal_gamma_pos": 0.0}, ""),
    ("L8_smooth",
     {"bce_weight": 0.5,  "soft_jaccard_weight": 1.0, "margin_weight": 0.05,
      "label_smoothing": 0.1, "use_focal": False}, ""),
    ("L9_atc",
     {"bce_weight": 0.5,  "soft_jaccard_weight": 1.0, "margin_weight": 0.05,
      "label_smoothing": 0.0, "use_focal": False},
     "--use_atc_coarse_loss --atc_loss_weight 0.1"),
    ("L10_lab_impute",
     {"bce_weight": 0.5,  "soft_jaccard_weight": 1.0, "margin_weight": 0.05,
      "label_smoothing": 0.0, "use_focal": False},
     "--use_lab_impute_loss"),
]
print(f"Ready: {len(LOSS_CONFIGS)} loss variants x {len(SEEDS)} seeds = {len(LOSS_CONFIGS)*len(SEEDS)} runs")

SEEDS = [42, 123, 456]
reports_dir = Path("experiment_reports/sweep_15a_loss")
results_log = []

def run_group(configs, group_tag):
    """configs = [(name, cfg_path, extra_flags), ...]"""
    import torch
    for name, cfg_path, extra_flags in configs:
        for seed in SEEDS:
            run_name = f"{name}_seed{seed}"
            run_dir  = reports_dir / run_name
            run_dir.mkdir(parents=True, exist_ok=True)
            if list(run_dir.glob("result_*.json")):
                print(f"  [SKIP] {run_name} — result already exists")
                results_log.append(f"SKIP: {run_name}")
                continue
            log_path = run_dir / "training_log.txt"
            cmd = (
                f"python -u src/train.py --config {cfg_path} "
                f"{BACKBONE_FLAGS} {extra_flags} "
                f"--seed {seed} --results_dir {run_dir}"
            )
            print(f"\n=== [{group_tag}] {run_name} ===")
            try:
                with open(log_path, "w") as lf:
                    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                            stderr=subprocess.STDOUT, text=True)
                    for line in proc.stdout:
                        print(line, end="")
                        lf.write(line)
                    proc.wait()
                status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
                results_log.append(f"{status}: {run_name}")
            except Exception as e:
                results_log.append(f"CRASH: {run_name}: {e}")
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"Config helpers ready. Reports -> {reports_dir}")


In [ ]:
# ── Smoke test: 1 epoch, 3 configs ───────────────────────────────────────────
Path("smoke_s15a").mkdir(exist_ok=True)
SMOKE_NAMES = ["L0_baseline", "L7_focal", "L9_atc"]
smoke_results = []
for cfg_name, train_ov, extra_cli in LOSS_CONFIGS:
    if cfg_name not in SMOKE_NAMES: continue
    cfg_path = f"s15a_smoke_{cfg_name}.yaml"
    make_config(cfg_path, model_overrides=BEST_MODEL,
                training_overrides=train_ov, preprocessing_overrides=FULL_PREP,
                smoke_epochs=1)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{BACKBONE_FLAGS} {LAB_FLAGS} {extra_cli} "
           f"--seed 42 --results_dir smoke_s15a/{cfg_name}")
    print(f"SMOKE {cfg_name}: {cmd}")
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    for ln in (proc.stdout + proc.stderr).strip().split("\n")[-5:]: print(ln)
    status = "PASS" if proc.returncode == 0 else f"FAIL ({proc.returncode})"
    smoke_results.append(f"{status}: {cfg_name}")
    print(f">>> {status}\n")
for r in smoke_results: print(r)
if any("FAIL" in r for r in smoke_results):
    raise RuntimeError("Smoke tests failed.")
print("All smoke tests passed.")


In [ ]:
# ── Full sweep — 11 variants × 3 seeds = 33 runs ─────────────────────────────
print("\n" + "#"*70 + "\n# LOSS FUNCTION SWEEP — full system\n" + "#"*70)
all_configs = []
for cfg_name, train_ov, extra_cli in LOSS_CONFIGS:
    cfg_path = f"s15a_{cfg_name}.yaml"
    make_config(cfg_path, model_overrides=BEST_MODEL,
                training_overrides=train_ov, preprocessing_overrides=FULL_PREP)
    flags = f"{LAB_FLAGS} {extra_cli}"
    all_configs.append((cfg_name, cfg_path, flags))

run_group(all_configs, "15a")
print("\nAll runs done.")


In [ ]:
# ── Results ──────────────────────────────────────────────────────────────────
import json, numpy as np

def get_metric(d, key):
    sec = d.get("test_metrics", {})
    for k in [key, key.replace("_", " "), key.title()]:
        if k in sec: return float(sec[k])
        if k in d:   return float(d[k])
    return 0.0

all_results = {}
for jp in sorted(reports_dir.rglob("result_*.json")):
    with open(jp) as f: d = json.load(f)
    run_name = jp.parent.name
    idx = run_name.rfind("_seed")
    if idx == -1: continue
    all_results.setdefault(run_name[:idx], []).append(d)

def summarize(name):
    runs = all_results.get(name, [])
    if not runs: return None
    jac = [get_metric(d, "jaccard")  for d in runs]
    f1  = [get_metric(d, "f1")       for d in runs]
    ddi = [get_metric(d, "DDI Rate") for d in runs]
    return dict(jac=np.mean(jac), std=np.std(jac, ddof=1) if len(jac)>1 else 0,
                f1=np.mean(f1), ddi=np.mean(ddi), n=len(jac))

print("\n" + "="*80)
print("SWEEP 15a — LOSS FUNCTION RESULTS (full system, 3 seeds)")
print("="*80)
print(f"  {'Config':<20} {'Jac mean±std':>16}  {'delta':>8}  F1      DDI     n")
print(f"  {'-'*20} {'-'*16}  {'-'*8}  {'-'*6}  {'-'*6}  -")

sums = [(name, summarize(name)) for name, _, _ in LOSS_CONFIGS]
valid = [s for _, s in sums if s]
best_jac = max(s["jac"] for s in valid) if valid else 0
ref_jac  = next((s["jac"] for name, s in sums if name == "L0_baseline" and s), 0)

for name, s in sums:
    if s is None:
        print(f"  {name:<20} (missing)")
        continue
    delta = f"{s['jac'] - ref_jac:+.4f}"
    mark  = " *" if abs(s["jac"] - best_jac) < 1e-6 else ""
    print(f"  {name:<20} {s['jac']:.4f}±{s['std']:.4f}  {delta:>8}  "
          f"{s['f1']:.4f}  {s['ddi']:.4f}  {s['n']}{mark}")
print("="*80)


In [ ]:
total   = len(results_log)
success = sum(1 for r in results_log if "SUCCESS" in r)
failed  = sum(1 for r in results_log if "FAILED"  in r)
crashed = sum(1 for r in results_log if "CRASH"   in r)
print(f"\nRun log: {total} total | {success} success | {failed} failed | {crashed} crashed")
if failed + crashed > 0:
    print("\nFailed/crashed runs:")
    for r in results_log:
        if "SUCCESS" not in r: print(f"  {r}")


In [ ]:
import zipfile
from pathlib import Path

zip_name = "reports_sweep_15a_loss.zip"
rd = reports_dir
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(rd.rglob("result_*.json")):
        zf.write(p, p.relative_to(rd.parent))
    for p in sorted(rd.rglob("training_log.txt")):
        zf.write(p, p.relative_to(rd.parent))
print(f"Exported -> {zip_name}")
